In [79]:
import pandas as pd
import numpy as np

In [80]:
movies_dataset = pd.read_csv("Datasets/movies.csv")
ratings_dataset = pd.read_csv("Datasets/ratings.csv")

In [81]:
tags_dataset = pd.read_csv("Datasets/tags.csv")

In [82]:
movies_dataset.head()

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


In [83]:
ratings_dataset.head()

,userId,movieId,rating,timestamp
0,1,1,4.0,964982703
1,1,3,4.0,964981247
2,1,6,4.0,964982224
3,1,47,5.0,964983815
4,1,50,5.0,964982931


In [84]:
tags_dataset.head()

,userId,movieId,tag,timestamp
0,2,60756,funny,1445714994
1,2,60756,Highly quotable,1445714996
2,2,60756,will ferrell,1445714992
3,2,89774,Boxing story,1445715207
4,2,89774,MMA,1445715200


In [85]:
movies_dataset.shape, ratings_dataset.shape, tags_dataset.shape

((9742, 3), (100836, 4), (3683, 4))

In [86]:
movies_dataset.isnull().sum()

movieId    0
title      0
genres     0
dtype: int64

In [87]:
ratings_dataset.isnull().sum()

userId       0
movieId      0
rating       0
timestamp    0
dtype: int64

In [88]:
movies_dataset[movies_dataset['movieId'] == 60756]

,movieId,title,genres
6801,60756,Step Brothers (2008),Comedy


In [89]:
movies_dataset['genres'] = movies_dataset['genres'].str.split('|')

In [90]:
movies_dataset.head()

,movieId,title,genres
0,1,Toy Story (1995),"[Adventure, Animation, Children, Comedy, Fantasy]"
1,2,Jumanji (1995),"[Adventure, Children, Fantasy]"
2,3,Grumpier Old Men (1995),"[Comedy, Romance]"
3,4,Waiting to Exhale (1995),"[Comedy, Drama, Romance]"
4,5,Father of the Bride Part II (1995),[Comedy]


In [91]:
unique_genres = movies_dataset['genres'].explode().unique()

print(unique_genres)
print("Number of unique genres:", len(unique_genres))

['Adventure' 'Animation' 'Children' 'Comedy' 'Fantasy' 'Romance' 'Drama'
 'Action' 'Crime' 'Thriller' 'Horror' 'Mystery' 'Sci-Fi' 'War' 'Musical'
 'Documentary' 'IMAX' 'Western' 'Film-Noir' '(no genres listed)']
Number of unique genres: 20


In [92]:
movies_dataset['genres'] = movies_dataset['genres'].apply(lambda x: [i.replace('-', '') for i in x])

In [93]:
def convert(movie_genre):
    movie_genre_list = movie_genre.split(" ")
    return movie_genre_list

In [94]:
unique_genres = movies_dataset['genres'].explode().unique()

print(unique_genres)
print("Number of unique genres:", len(unique_genres))

['Adventure' 'Animation' 'Children' 'Comedy' 'Fantasy' 'Romance' 'Drama'
 'Action' 'Crime' 'Thriller' 'Horror' 'Mystery' 'SciFi' 'War' 'Musical'
 'Documentary' 'IMAX' 'Western' 'FilmNoir' '(no genres listed)']
Number of unique genres: 20


In [100]:
movies_dataset = movies_dataset[
    ~movies_dataset['genres'].apply(lambda x: '(no genres listed)' in x)
]

In [101]:
unique_genres = movies_dataset['genres'].explode().unique()

print(unique_genres)
print("Number of unique genres:", len(unique_genres))

['Adventure' 'Animation' 'Children' 'Comedy' 'Fantasy' 'Romance' 'Drama'
 'Action' 'Crime' 'Thriller' 'Horror' 'Mystery' 'SciFi' 'War' 'Musical'
 'Documentary' 'IMAX' 'Western' 'FilmNoir']
Number of unique genres: 19


In [102]:
movies_dataset['genres'] = movies_dataset['genres'].apply(lambda x: [i.replace(' ', '') for i in x])

In [104]:
movies_dataset.tail(5)

,movieId,title,genres
9737,193581,Black Butler: Book of the Atlantic (2017),"[Action, Animation, Comedy, Fantasy]"
9738,193583,No Game No Life: Zero (2017),"[Animation, Comedy, Fantasy]"
9739,193585,Flint (2017),[Drama]
9740,193587,Bungo Stray Dogs: Dead Apple (2018),"[Action, Animation]"
9741,193609,Andrew Dice Clay: Dice Rules (1991),[Comedy]


In [108]:
movies_dataset['genres'] = movies_dataset['genres'].apply(lambda x: " ".join(x))

In [110]:
movies_dataset.head()

,movieId,title,genres
0,1,Toy Story (1995),Adventure Animation Children Comedy Fantasy
1,2,Jumanji (1995),Adventure Children Fantasy
2,3,Grumpier Old Men (1995),Comedy Romance
3,4,Waiting to Exhale (1995),Comedy Drama Romance
4,5,Father of the Bride Part II (1995),Comedy


In [111]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [112]:
tfidf = TfidfVectorizer()

In [113]:
tfidf_matrix = tfidf.fit_transform(movies_dataset['genres'])

In [114]:
tfidf_matrix.shape

(9708, 19)

In [115]:
features = tfidf.get_feature_names_out()

print(features)

['action' 'adventure' 'animation' 'children' 'comedy' 'crime'
 'documentary' 'drama' 'fantasy' 'filmnoir' 'horror' 'imax' 'musical'
 'mystery' 'romance' 'scifi' 'thriller' 'war' 'western']


In [116]:
from sklearn.metrics.pairwise import cosine_similarity

In [117]:
cosine_sim = cosine_similarity(tfidf_matrix)

In [118]:
print(cosine_sim.shape)

(9708, 9708)


In [119]:
cosine_sim[0]

array([1.        , 0.8136036 , 0.15259961, ..., 0.        , 0.42114166,
       0.26738778], shape=(9708,))

In [120]:
indices = pd.Series(movies_dataset.index, index=movies_dataset['title'])

In [122]:
indices.head()

title
Toy Story (1995)                      0
Jumanji (1995)                        1
Grumpier Old Men (1995)               2
Waiting to Exhale (1995)              3
Father of the Bride Part II (1995)    4
dtype: int64

In [124]:
def recommend_movie(movie_title):
    movie_index = indices[movie_title]
    similarity_scores = list(enumerate(cosine_sim[movie_index]))

    similarity_scores = sorted(
        similarity_scores,
        key=lambda x: x[1],
        reverse=True
    )

    similarity_scores = similarity_scores[1:6]
    movie_indices = [i[0] for i in similarity_scores]
    return movies_dataset['title'].iloc[movie_indices]

In [125]:
recommend_movie("Toy Story (1995)")

1706                                       Antz (1998)
2355                                Toy Story 2 (1999)
2809    Adventures of Rocky and Bullwinkle, The (2000)
3000                  Emperor's New Groove, The (2000)
3568                             Monsters, Inc. (2001)
Name: title, dtype: object

In [126]:
import joblib

In [127]:
joblib.dump(cosine_sim, "similarity.joblib")
joblib.dump(movies_dataset, "movies.joblib")

['movies.joblib']